# 02b -- Channel / Unit Validation

For every included run and canonical signal: which raw column it resolved to,
the observed value range, and whether that range is physically plausible.
Surfaces silent mapping problems (missing channel, constant signal,
out-of-band value) *before* they reach the model.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
%matplotlib inline
import pdm_common as P

from pathlib import Path
P.RAW_DIR = Path(r"E:\DRDO\Sarthak Dhaigude_DRDO\PdM-main\RAW_DIR\raw")

import numpy as np, pandas as pd

In [2]:
UNIT_LABEL = {"pressure": "bar", "current": "A", "vdc": "V", "vac": "V",
              "speed": "rpm", "torque": "Nm", "flow": "Lpm", "fuel": "kg/s"}
# Sanity bands, NOT decision thresholds -- a value outside just raises a warning.
PLAUSIBLE = {"pressure": (0, 5000), "current": (-1000, 1000), "vdc": (-5, 400),
             "vac": (-1000, 1000), "speed": (-1e5, 1e5), "torque": (-1e4, 1e4),
             "flow": (-1e5, 1e5), "fuel": (0, 1)}
INFERRED_RAW = {"ps-simulink converter4", "ps-simulink converter9",
                "ps-simulink converter6", "ps-simulink converter16", "mass_flow_rate"}

rows = []
for fname, spec in P.FILE_MAP.items():
    path = P.RAW_DIR / fname
    if not path.exists():
        rows.append(dict(file=fname, canonical="-", raw_found="(FILE MISSING)",
                         expected_unit="-", observed_min="-", observed_max="-",
                         decision="missing", warning="file not found on disk"))
        continue
    raw = P.read_run(path, spec)
    harm = P.harmonize(raw, spec)
    for canon in P.CANON:
        mapped_raw = spec["cols"].get(canon)
        if mapped_raw is None:
            if canon in P.COMMON:
                rows.append(dict(file=fname, canonical=canon, raw_found="(none)",
                    expected_unit=UNIT_LABEL[canon], observed_min="-", observed_max="-",
                    decision="gated/absent", warning="in COMMON but not mapped for this run"))
            continue
        actual = P._resolve(mapped_raw, raw.columns)
        if actual is None or canon not in harm.columns:
            rows.append(dict(file=fname, canonical=canon, raw_found=f"{mapped_raw} (UNRESOLVED)",
                expected_unit=UNIT_LABEL[canon], observed_min="-", observed_max="-",
                decision="broken", warning="mapped raw column not found in file"))
            continue
        v = pd.to_numeric(harm[canon], errors="coerce").to_numpy()
        v = v[np.isfinite(v)]
        vmin, vmax = (float(v.min()), float(v.max())) if v.size else (np.nan, np.nan)
        warns = []
        if v.size == 0:
            warns.append("all-NaN")
        elif np.nanstd(v) == 0:
            warns.append("constant (no information)")
        lo, hi = PLAUSIBLE[canon]
        if v.size and (vmax < lo or vmin > hi):
            warns.append(f"outside plausible {UNIT_LABEL[canon]} band {lo}..{hi}")
        decision = "inferred" if P._norm(actual) in INFERRED_RAW else "direct"
        rows.append(dict(file=fname, canonical=canon, raw_found=actual,
            expected_unit=UNIT_LABEL[canon], observed_min=round(vmin, 4),
            observed_max=round(vmax, 4), decision=decision, warning="; ".join(warns)))

rep = pd.DataFrame(rows)
rep.to_csv(P.ART_DIR / "channel_validation.csv", index=False)
order = {s: i for i, s in enumerate(P.COMMON)}
rep["_o"] = rep["canonical"].map(lambda c: order.get(c, 99))
rep = rep.sort_values(["file", "_o", "canonical"]).drop(columns="_o").reset_index(drop=True)
warned = rep[rep["warning"].astype(bool) & (rep["warning"] != "")]
print(f"{len(warned)} row(s) with warnings out of {len(rep)}.")
rep

2 row(s) with warnings out of 94.


,file,canonical,raw_found,expected_unit,observed_min,observed_max,decision,warning
0,Healthy Data 3.xlsx,pressure,pressure,bar,1.0132,1.0132,direct,
1,Healthy Data 3.xlsx,current,bus_current,A,-0.0000,5.5291,direct,
2,Healthy Data 3.xlsx,vdc,bus_voltage,V,-0.0000,27.6453,direct,
3,Healthy Data 3.xlsx,vac,"line_voltage(1,1)",V,-753.5239,683.4447,direct,
4,Healthy Data 3.xlsx,flow,flow_rate,Lpm,-0.0000,89.1164,direct,
...,...,...,...,...,...,...,...,...
89,simplifiied-generator-fault.xlsx,vac,Voltage,V,-187.5521,187.5607,direct,
90,simplifiied-generator-fault.xlsx,flow,Pump_Flow_volume,Lpm,-9.5106,0.0000,direct,
91,simplifiied-generator-fault.xlsx,fuel,Fuel_Consumption,kg/s,0.0000,0.0000,direct,constant (no information)
92,simplifiied-generator-fault.xlsx,speed,Angular_Velocity,rpm,-16173.3068,4.3285,direct,
